In [23]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
import operator

In [24]:
load_dotenv()

True

In [25]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [26]:
# model.invoke("What is the capital of India? In one sentence").content

In [27]:
class SentimentSchema(BaseModel):
    sentiment : Literal["positive","negative"] = Field(description="Sentiment of the review.")
    
class DiagnosisSchema(BaseModel):
    issue_type:Literal["UI","Billing","Bug","Performance","Support","Other"] = Field(description="The category of the issue mentioned in the review.")
    tone:Literal["Angry","Frustrated","Dissapointed","Calm"]=Field(description="The emotional tone expressed by the user") 
    urgency:Literal["Low","Medium","High"] = Field(description="The urgency or criticality of the issue based on the review")   

In [28]:
sentiment_model = model.with_structured_output(SentimentSchema)
diagnosis_model = model.with_structured_output(DiagnosisSchema)

In [29]:
# sentiment_model.invoke("The UI is not loading, very slow. But the functionality is great!")

In [30]:
class ReviewState(TypedDict):
    review : str
    sentiment : Literal["positive","negative"]
    diagnosis : dict
    response : str    

In [71]:
graph = StateGraph(ReviewState)

In [72]:
def find_sentiment(state:ReviewState):
    prompt = f"What is the sentiment of the review? {state['review']}"
    sentiment = sentiment_model.invoke(prompt).sentiment
    return {'sentiment':sentiment}

def check_sentiment(state:ReviewState) -> Literal["positive_review", "diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_review'
    else: 
        return 'diagnosis'

def diagnosis(state:ReviewState):
    prompt = f"Diagnose this negative review: \n{state['review']} \nand return issue_type, tone and urgency"
    diagnosis = diagnosis_model.invoke(prompt)
    return {'diagnosis':diagnosis.model_dump()}

def positive_review(state:ReviewState):
    prompt = f"""
    You are a support assistant replying to a user. Write a warm thank you message in response to this review: {state['review']}\n
    Also ask the user to leave feedback on our website. Keep the message concise.
    """
    response = model.invoke(prompt).content
    return {'response':response}

def negative_review(state:ReviewState):
    diagnosis = state['diagnosis']
    prompt = f"""
    You are a support assistant replying to a user. The user has {diagnosis['issue_type']} issue, the user sounded {diagnosis['tone']} and marked urgency as {diagnosis['urgency']}.
    Write a empathetic and helpful resolution message for the user. Keep it concise.
    """
    response = model.invoke(prompt).content
    return {'response':response}



In [73]:
graph.add_node("find_sentiment", find_sentiment)
graph.add_node("diagnosis", diagnosis)
graph.add_node("positive_review", positive_review)
graph.add_node("negative_review", negative_review)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('diagnosis', 'negative_review')
graph.add_edge('positive_review', END)
graph.add_edge('negative_review', END) 

In [74]:
workflow = graph.compile()

In [78]:
workflow.get_graph().print_ascii()

                  +-----------+                    
                  | __start__ |                    
                  +-----------+                    
                        *                          
                        *                          
                        *                          
                +----------------+                 
                | find_sentiment |                 
                +----------------+                 
                ...            ...                 
              ..                  ...              
            ..                       ..            
   +-----------+                       ..          
   | diagnosis |                        .          
   +-----------+                        .          
          *                             .          
          *                             .          
          *                             .          
+-----------------+           +-----------------+  
| negative_r

In [ ]:
initial_state = {'review':"I have been using this app for about a month now. The UI is really impressive. Everything is within reach and quick to response. There is literally no need of any document for SOP."}
# initial_state = {'review':"I have been using this app for about a month now. I am sad to say the app is not serving the purpose. Its really slow and times gets stuck during payment. I am really frustrated and need to resolve my payment issue on priority."}

In [77]:
workflow.invoke(initial_state)

{'review': 'I have been using this app for about a month now. The UI is reallly impressive. Everything is within reach and quick to response. There is literally no need of any document for SOP.',
 'sentiment': 'positive',
 'response': "Thank you so much for your wonderful review! We're thrilled to hear you're enjoying the app and find the UI impressive and responsive. It's fantastic to know it's making things so easy for you!\n\nWe'd love to hear more about your experience. Please consider leaving additional feedback on our website at [Your Website Link Here]!"}